In [16]:
import torch
import torch.nn as nn
#pde: u_tt = c^2 * u_xx
class pinn(nn.Module):
    def __init__(self, c=1.0, lr=0.001):
        super().__init__()
        self.c = c

        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )

        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)

    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        return self.net(inputs)

        #loss function
    def loss(self, x, t):
        x.requires_grad_(True)
        t.requires_grad_(True)

        u = self.forward(x, t)

        #diff
        u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
        u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]

        u_tt = torch.autograd.grad(u_t, t, grad_outputs=torch.ones_like(u_t), create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]


        err = u_tt - (self.c**2) * u_xx
        return torch.mean(err**2)

        #training function
    def train_model(self, epochs=1000, num_points=500):

        for epoch in range(epochs):
            self.optimizer.zero_grad()

            x_col = torch.rand(num_points, 1) * 2.0 - 1.0
            t_col = torch.rand(num_points, 1) * 1.0

            loss_val = self.loss(x_col, t_col)

            loss_val.backward()
            self.optimizer.step()

            if epoch % 200 == 0:
                print(f"Epoch {epoch:4d}  Loss: {loss_val.item():.6f}")

        print("Training Completed")

#objects
if __name__ == "__main__":
    print("\n First model\n")
    model1 = pinn(c=1.0, lr=0.001)
    model1.train_model(epochs=1000, num_points=500)

    print("Second model (c=2)\n")
    model2=pinn(c=2.0, lr=.001)
    model2.train(epochs=1000,num_points=500)

    print("Third model aggressive learning (lr=.01,epochs=300)\n")
    model3=pinn(c=1.0,lr=.01)
    model3.train(epochs=300,num_points=500)



 First model

Epoch    0,  Loss: 0.000226
Epoch  200,  Loss: 0.000001
Epoch  400,  Loss: 0.000000
Epoch  600,  Loss: 0.000000
Epoch  800,  Loss: 0.000000
Training Completed

Second model aggressive learning (lr=.01,epochs=300)

Epoch    0,  Loss: 0.001451
Epoch  200,  Loss: 0.000000
Training Completed
